# Model Training

This notebook trains recommendation models using features from the retailrocket dataset.

Run notebook 03 first if features haven't been generated yet.

In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import ndcg_score, label_ranking_average_precision_score
from scipy.sparse import csr_matrix
from pathlib import Path
import json
import pickle
from datetime import datetime
from collections import defaultdict
import os
from pprint import pprint

print(f"pandas: {pd.__version__}")
print(f"numpy: {np.__version__}")
print(f"lightgbm: {lgb.__version__}")

pandas: 2.3.3
numpy: 2.3.5
lightgbm: 4.6.0


## Training Data Strategy

Models are trained on the RetailRocket dataset (2.7M events, 1.4M users, 235k products). Synthetic data is used only for infrastructure testing.

In [2]:
# Configuration
DATA_MODE = "retailrocket"
features_dir = Path(f'artifacts/features/{DATA_MODE}')

# Load feature tables
user_features = pd.read_parquet(features_dir / 'user_features.parquet')
item_features = pd.read_parquet(features_dir / 'item_features.parquet')
interaction_features = pd.read_parquet(features_dir / 'interaction_features.parquet')

print(f"Loaded features from: {features_dir}")
print(f"User features: {user_features.shape}")
print(f"Item features: {item_features.shape}")
print(f"Interaction features: {interaction_features.shape}")

Loaded features from: artifacts\features\retailrocket
User features: (1407580, 9)
Item features: (235061, 8)
Interaction features: (2145179, 6)


In [3]:
# Load original events for label engineering
if DATA_MODE == "retailrocket":
    events_path = Path('artifacts/external/retailrocket/events.parquet')
elif DATA_MODE == "synthetic":
    events_path = Path('artifacts/events')
    parquet_files = list(events_path.rglob('*.parquet'))
    df_events = pd.concat([pd.read_parquet(f) for f in parquet_files], ignore_index=True)
elif DATA_MODE == "merged":
    events_path = Path('artifacts/combined/events.parquet')
else:
    raise ValueError(f"Invalid DATA_MODE: {DATA_MODE}")

if DATA_MODE != "synthetic":
    df_events = pd.read_parquet(events_path)

print(f"Loaded events: {df_events.shape}")
print(f"Event types: {df_events['event_type'].unique()}")

Loaded events: (2756101, 7)
Event types: ['view' 'add_to_cart' 'purchase']


In [4]:
# Merge interaction features with user and item features
df_train = interaction_features.copy()

df_train = df_train.merge(
    user_features.add_prefix('user_'),
    left_on='user_id',
    right_on='user_user_id',
    how='left'
)

df_train = df_train.merge(
    item_features.add_prefix('item_'),
    left_on='product_id',
    right_on='item_product_id',
    how='left'
)

df_train = df_train.drop(columns=['user_user_id', 'item_product_id'])

print(f"Merged training data: {df_train.shape}")

Merged training data: (2145179, 21)


## Label Engineering

Relevance labels based on event types:
- Purchase = 3
- Add to Cart = 2
- View = 1

In [5]:
# Create relevance labels from event types
event_labels = df_events.copy()
event_labels['relevance'] = event_labels['event_type'].map({
    'view': 1,
    'add_to_cart': 2,
    'purchase': 3
})

# Take maximum relevance per user-product pair
labels = event_labels.groupby(['user_id', 'product_id'])['relevance'].max().reset_index()

print(f"Labels created: {labels.shape}")
print(f"Label distribution:\n{labels['relevance'].value_counts().sort_index()}")

Labels created: (2145179, 3)
Label distribution:
relevance
1    2080929
2      42980
3      21270
Name: count, dtype: int64


In [6]:
# Merge labels into training data
df_train = df_train.merge(labels, on=['user_id', 'product_id'], how='left')

print(f"Missing labels: {df_train['relevance'].isna().sum()}")
print(f"Final training data: {df_train.shape}")
df_train.head(3)

Missing labels: 0
Final training data: (2145179, 22)


,user_id,product_id,interaction_count,last_interaction_ts,has_purchased,recency_days,user_total_events,user_unique_products_interacted,user_unique_sessions,user_last_event_ts,...,user_views_count,user_recency_days,item_last_interaction_ts,item_total_add_to_cart,item_total_purchases,item_total_views,item_popularity_score,item_conversion_rate,item_recency_days,relevance
0,0,285930,1,2015-09-11 20:49:49+00:00,0,6.256921,3,3,1,2015-09-11 20:55:17+00:00,...,3,6.253125,2015-09-16 00:19:31+00:00,5,0,90,4.564348,0.000000,2.111296,1
1,0,357564,1,2015-09-11 20:52:39+00:00,0,6.254954,3,3,1,2015-09-11 20:55:17+00:00,...,3,6.253125,2015-09-17 18:08:12+00:00,2,2,267,5.605802,0.007491,0.369155,1
2,0,67045,1,2015-09-11 20:55:17+00:00,0,6.253125,3,3,1,2015-09-11 20:55:17+00:00,...,3,6.253125,2015-09-17 20:01:32+00:00,2,1,202,5.327876,0.004950,0.290451,1


## Train / Validation Split

Time-based split to avoid data leakage: train on past interactions, validate on future.

In [7]:
# Time-based split using last_interaction_ts
split_percentile = 80
split_timestamp = df_train['last_interaction_ts'].quantile(split_percentile / 100)

train_mask = df_train['last_interaction_ts'] <= split_timestamp
val_mask = df_train['last_interaction_ts'] > split_timestamp

df_train_set = df_train[train_mask].copy()
df_val_set = df_train[val_mask].copy()

print(f"Split timestamp: {split_timestamp}")
print(f"Train set: {df_train_set.shape[0]} interactions ({train_mask.sum() / len(df_train) * 100:.1f}%)")
print(f"Val set: {df_val_set.shape[0]} interactions ({val_mask.sum() / len(df_train) * 100:.1f}%)")

Split timestamp: 2015-08-18 20:48:50+00:00
Train set: 1716144 interactions (80.0%)
Val set: 429035 interactions (20.0%)


## Baseline: Popularity-Based Recommender

In [8]:
# Calculate popularity scores from training set
popularity_scores = df_train_set.groupby('product_id')['relevance'].sum().sort_values(ascending=False)

print(f"Top 10 most popular products:")
print(popularity_scores.head(10))

Top 10 most popular products:
product_id
5411      1925
187946    1587
370653    1376
461686    1206
298009    1180
309778    1107
257040    1073
96924     1073
335975    1050
219512     980
Name: relevance, dtype: int64


In [9]:
# Evaluate popularity baseline on validation set (optimized)
def evaluate_popularity_baseline(val_set, popularity_scores, k=10, max_users=5000):
    """
    Evaluate popularity baseline using NDCG@k.
    For each user, predict top-k popular items and compare to their actual interactions.
    
    Args:
        val_set: Validation dataset
        popularity_scores: Product popularity scores
        k: Number of top items to recommend
        max_users: Maximum number of users to evaluate (for speed)
    """
    # Pre-compute top-k popular products (same for all users)
    top_k_products = popularity_scores.head(k).index.tolist()
    top_k_scores = popularity_scores.head(k).values
    
    # Sample users if dataset is too large
    unique_users = val_set['user_id'].unique()
    if len(unique_users) > max_users:
        print(f"Sampling {max_users} users from {len(unique_users)} for faster evaluation")
        sampled_users = np.random.choice(unique_users, size=max_users, replace=False)
        val_set_sample = val_set[val_set['user_id'].isin(sampled_users)]
    else:
        val_set_sample = val_set
    
    # Group by user for efficient iteration
    ndcg_scores = []
    for user_id, user_data in val_set_sample.groupby('user_id'):
        # Skip users with only 1 interaction (NDCG requires at least 2 documents)
        if len(user_data) < 2:
            continue
        
        # Get ground truth relevance scores
        true_relevance = user_data.set_index('product_id')['relevance'].to_dict()
        
        # Build prediction vector (relevance score for each predicted item)
        y_true = [true_relevance.get(pid, 0) for pid in top_k_products]
        y_pred = top_k_scores
        
        # NDCG requires at least one non-zero relevance
        if sum(y_true) > 0:
            ndcg = ndcg_score([y_true], [y_pred], k=k)
            ndcg_scores.append(ndcg)
    
    print(f"Evaluated {len(ndcg_scores)} users")
    return np.mean(ndcg_scores) if ndcg_scores else 0.0

baseline_ndcg = evaluate_popularity_baseline(df_val_set, popularity_scores, k=10, max_users=5000)
print(f"\nPopularity Baseline NDCG@10: {baseline_ndcg:.4f}")

Sampling 5000 users from 302355 for faster evaluation
Evaluated 16 users

Popularity Baseline NDCG@10: 0.4234


## Collaborative Filtering: SVD Matrix Factorization

In [10]:
# Create sparse user-item interaction matrix
print("Building sparse interaction matrix...")

# Create mappings for user and product IDs (include train + val for mapping)
all_user_ids = pd.concat([df_train_set['user_id'], df_val_set['user_id']]).unique()
all_product_ids = pd.concat([df_train_set['product_id'], df_val_set['product_id']]).unique()

user_id_to_idx = {uid: idx for idx, uid in enumerate(all_user_ids)}
product_id_to_idx = {pid: idx for idx, pid in enumerate(all_product_ids)}

# Map IDs to indices (only for training data)
row_indices = df_train_set['user_id'].map(user_id_to_idx).values
col_indices = df_train_set['product_id'].map(product_id_to_idx).values
data = df_train_set['relevance'].values

# Create sparse matrix (CSR format)
interaction_matrix_sparse = csr_matrix(
    (data, (row_indices, col_indices)),
    shape=(len(all_user_ids), len(all_product_ids))
)

print(f"Shape: {interaction_matrix_sparse.shape} (users × products)")
print(f"Non-zero entries: {interaction_matrix_sparse.nnz:,}")
print(f"Sparsity: {100 * (1 - interaction_matrix_sparse.nnz / (interaction_matrix_sparse.shape[0] * interaction_matrix_sparse.shape[1])):.4f}%")

Building sparse interaction matrix...
Shape: (1407580, 235061) (users × products)
Non-zero entries: 1,716,144
Sparsity: 99.9995%


In [11]:
# Train SVD model
print("Training SVD model...")
n_components = 10
svd_model = TruncatedSVD(n_components=n_components, random_state=42)
user_factors = svd_model.fit_transform(interaction_matrix_sparse)
item_factors = svd_model.components_.T

print(f"User factors: {user_factors.shape}")
print(f"Item factors: {item_factors.shape}")
print(f"Explained variance: {svd_model.explained_variance_ratio_.sum():.4f}")

Training SVD model...
User factors: (1407580, 10)
Item factors: (235061, 10)
Explained variance: 0.0167


In [12]:
# Evaluate SVD on validation set
def evaluate_svd(val_set, user_id_to_idx, product_id_to_idx, user_factors, item_factors, k=10, max_users=5000):
    idx_to_product_id = {idx: pid for pid, idx in product_id_to_idx.items()}
    
    unique_users = val_set['user_id'].unique()
    if len(unique_users) > max_users:
        print(f"Sampling {max_users} users from {len(unique_users)} for evaluation")
        sampled_users = np.random.choice(unique_users, size=max_users, replace=False)
        val_set_sample = val_set[val_set['user_id'].isin(sampled_users)]
    else:
        val_set_sample = val_set
    
    ndcg_scores = []
    cold_start_users = 0
    
    for user_id, user_data in val_set_sample.groupby('user_id'):
        if user_id not in user_id_to_idx:
            cold_start_users += 1
            continue
        
        if len(user_data) < 2:
            continue
        
        true_relevance = user_data.set_index('product_id')['relevance'].to_dict()
        
        user_idx = user_id_to_idx[user_id]
        user_vec = user_factors[user_idx]
        pred_scores = user_vec @ item_factors.T
        
        top_k_indices = np.argsort(pred_scores)[::-1][:k]
        top_k_products = [idx_to_product_id[idx] for idx in top_k_indices]
        
        y_true = [true_relevance.get(pid, 0) for pid in top_k_products]
        y_pred = [pred_scores[idx] for idx in top_k_indices]
        
        if sum(y_true) > 0:
            ndcg = ndcg_score([y_true], [y_pred], k=k)
            ndcg_scores.append(ndcg)
    
    print(f"Evaluated {len(ndcg_scores)} users ({cold_start_users} cold-start skipped)")
    return np.mean(ndcg_scores) if ndcg_scores else 0.0

svd_ndcg = evaluate_svd(df_val_set, user_id_to_idx, product_id_to_idx, user_factors, item_factors, k=10, max_users=5000)
print(f"SVD NDCG@10: {svd_ndcg:.4f}")

Sampling 5000 users from 302355 for evaluation
Evaluated 4 users (0 cold-start skipped)
SVD NDCG@10: 0.6835


## LightGBM Ranker

In [13]:
# Prepare feature columns (exclude IDs, timestamps, and labels)
exclude_cols = ['user_id', 'product_id', 'last_interaction_ts', 'user_last_event_ts', 'item_last_interaction_ts', 'relevance']
feature_cols = [col for col in df_train_set.columns if col not in exclude_cols]

print(f"Features for LightGBM: {len(feature_cols)}")
print(f"Feature columns: {feature_cols}")

Features for LightGBM: 16
Feature columns: ['interaction_count', 'has_purchased', 'recency_days', 'user_total_events', 'user_unique_products_interacted', 'user_unique_sessions', 'user_add_to_cart_count', 'user_purchase_count', 'user_views_count', 'user_recency_days', 'item_total_add_to_cart', 'item_total_purchases', 'item_total_views', 'item_popularity_score', 'item_conversion_rate', 'item_recency_days']


In [14]:
# Prepare training data for LightGBM Ranker
X_train = df_train_set[feature_cols].apply(pd.to_numeric, errors='coerce').fillna(0)
y_train = df_train_set['relevance']
group_train = df_train_set.groupby('user_id').size().values

X_val = df_val_set[feature_cols].apply(pd.to_numeric, errors='coerce').fillna(0)
y_val = df_val_set['relevance']
group_val = df_val_set.groupby('user_id').size().values

print(f"X_train: {X_train.shape}, groups: {len(group_train)}")
print(f"X_val:   {X_val.shape}, groups: {len(group_val)}")

X_train: (1716144, 16), groups: 1127390
X_val:   (429035, 16), groups: 302355


In [15]:
# Train LightGBM Ranker
lgb_train = lgb.Dataset(X_train, y_train, group=group_train)
lgb_val = lgb.Dataset(X_val, y_val, group=group_val, reference=lgb_train)

params = {
    'objective': 'lambdarank',
    'metric': 'ndcg',
    'ndcg_eval_at': [10],
    'learning_rate': 0.05,
    'num_leaves': 31,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': -1,
    'seed': 42
}

print("Training LightGBM Ranker...")
lgb_model = lgb.train(
    params,
    lgb_train,
    num_boost_round=100,
    valid_sets=[lgb_train, lgb_val],
    valid_names=['train', 'val']
)

print("\nTraining complete!")

Training LightGBM Ranker...

Training complete!


In [16]:
# Extract feature importance
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': lgb_model.feature_importance(importance_type='gain')
}).sort_values('importance', ascending=False)

print("\nTop 10 Most Important Features:")
print(feature_importance.head(10))


Top 10 Most Important Features:
                            feature    importance
0                 interaction_count  87789.047697
1                     has_purchased  38329.732391
10           item_total_add_to_cart  25806.450904
7               user_purchase_count  18222.838638
8                  user_views_count   8145.766548
12                 item_total_views   5549.336007
4   user_unique_products_interacted   3749.008644
13            item_popularity_score   3198.539872
3                 user_total_events   2583.462591
14             item_conversion_rate   1890.426016


In [17]:
# Evaluate LightGBM with multiple ranking metrics
def evaluate_lightgbm(model, X_val, y_val, group_val, k=10):
    """
    Evaluate LightGBM ranker with NDCG@k, Recall@k, and Precision@k.
    """
    y_pred = model.predict(X_val)
    
    ndcg_scores = []
    recall_scores = []
    precision_scores = []
    
    start_idx = 0
    for group_size in group_val:
        end_idx = start_idx + group_size
        
        y_true_group = y_val.iloc[start_idx:end_idx].values
        y_pred_group = y_pred[start_idx:end_idx]
        
        # Skip groups with only 1 item (NDCG requires at least 2 documents)
        if len(y_true_group) < 2:
            start_idx = end_idx
            continue
        
        # Get top-k predictions
        top_k_indices = np.argsort(y_pred_group)[::-1][:k]
        
        # NDCG@k (requires at least one relevant item)
        if np.sum(y_true_group) > 0:
            ndcg = ndcg_score([y_true_group], [y_pred_group], k=k)
            ndcg_scores.append(ndcg)
        
        # Recall@k and Precision@k: relevant items are those with relevance > 1 (cart or purchase)
        relevant_items = np.where(y_true_group > 1)[0]
        if len(relevant_items) > 0:
            # Recall: fraction of relevant items found in top-k
            recall = len(set(top_k_indices) & set(relevant_items)) / len(relevant_items)
            recall_scores.append(recall)
            
            # Precision: fraction of top-k that are relevant
            precision = len(set(top_k_indices) & set(relevant_items)) / min(k, len(top_k_indices))
            precision_scores.append(precision)
        
        start_idx = end_idx
    
    return {
        'ndcg@10': np.mean(ndcg_scores) if ndcg_scores else 0,
        'recall@10': np.mean(recall_scores) if recall_scores else 0,
        'precision@10': np.mean(precision_scores) if precision_scores else 0
    }

lgb_metrics = evaluate_lightgbm(lgb_model, X_val, y_val, group_val, k=10)

print("\nLightGBM Ranking Metrics:")
for metric, value in lgb_metrics.items():
    print(f"  {metric}: {value:.4f}")


LightGBM Ranking Metrics:
  ndcg@10: 0.9992
  recall@10: 0.9908
  precision@10: 0.4728


## Model Comparison

In [18]:
# Compare all models
comparison = pd.DataFrame({
    'Model': ['Popularity Baseline', 'SVD (Collaborative Filtering)', 'LightGBM Ranker'],
    'NDCG@10': [baseline_ndcg, svd_ndcg, lgb_metrics['ndcg@10']],
    'Recall@10': [None, None, lgb_metrics['recall@10']],
    'Precision@10': [None, None, lgb_metrics['precision@10']]
})

print("\nModel Comparison:")
print(comparison.to_string(index=False))


Model Comparison:
                        Model  NDCG@10  Recall@10  Precision@10
          Popularity Baseline 0.423440        NaN           NaN
SVD (Collaborative Filtering) 0.683485        NaN           NaN
              LightGBM Ranker 0.999232   0.990792      0.472798


### Interpretation

Popularity baseline provides weak performance due to lack of personalization. SVD improves by learning user preferences. LightGBM performs best by leveraging all engineered features and optimizing ranking loss directly.

In [19]:
# Create models directory
models_dir = Path('artifacts/models')
models_dir.mkdir(parents=True, exist_ok=True)

# Save LightGBM model
model_path = models_dir / 'lightgbm_ranker.txt'
lgb_model.save_model(str(model_path))
print(f"Model saved: {model_path}")

# Save model metadata
metadata = {
    'model_type': 'LightGBM Ranker',
    'trained_at': datetime.now().isoformat(),
    'features': feature_cols,
    'metrics': {
        'ndcg@10': float(lgb_metrics['ndcg@10']),
        'recall@10': float(lgb_metrics['recall@10']),
        'precision@10': float(lgb_metrics['precision@10'])
    },
    'hyperparameters': params,
    'num_boost_round': 100,
    'training_samples': len(X_train),
    'validation_samples': len(X_val)
}

metadata_path = models_dir / 'model_metadata.json'
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)
    
print(f"Metadata saved: {metadata_path}")

Model saved: artifacts\models\lightgbm_ranker.txt
Metadata saved: artifacts\models\model_metadata.json


In [20]:
# Also save SVD model for comparison purposes
svd_artifacts = {
    'model': svd_model,
    'user_factors': user_factors,
    'item_factors': item_factors,
    'user_id_to_idx': user_id_to_idx,
    'product_id_to_idx': product_id_to_idx
}

svd_path = models_dir / 'svd_model.pkl'
with open(svd_path, 'wb') as f:
    pickle.dump(svd_artifacts, f)
    
print(f"SVD model saved: {svd_path}")

SVD model saved: artifacts\models\svd_model.pkl


In [21]:
# Save feature importance for analysis
feature_importance_path = models_dir / 'feature_importance.csv'
feature_importance.to_csv(feature_importance_path, index=False)
print(f"Feature importance saved: {feature_importance_path}")

print("\nAll models exported successfully")

Feature importance saved: artifacts\models\feature_importance.csv

All models exported successfully


## Item-Item Co-Visitation Similarity

In [22]:
# Build item-item co-visitation matrix (optimized)
print("Building item-item co-visitation similarity...")

# Group by user session (user_id) and collect viewed items
user_sessions = df_train_set.groupby('user_id')['product_id'].apply(list).values

# Count co-occurrences with session size limit to prevent infinite loops
covisit_counts = defaultdict(lambda: defaultdict(int))
item_counts = defaultdict(int)

MAX_SESSION_SIZE = 50  # Limit session size to prevent O(n^2) explosion
sessions_processed = 0
sessions_skipped = 0

for session in user_sessions:
    # Skip sessions that are too large (likely data quality issues)
    if len(session) > MAX_SESSION_SIZE:
        sessions_skipped += 1
        continue
    
    sessions_processed += 1
    
    # Increment item counts
    for item in session:
        item_counts[item] += 1
    
    # Increment co-occurrence counts (only for reasonably sized sessions)
    for i, item1 in enumerate(session):
        for item2 in session[i+1:]:
            covisit_counts[item1][item2] += 1
            covisit_counts[item2][item1] += 1
    
    # Progress indicator
    if sessions_processed % 10000 == 0:
        print(f"  Processed {sessions_processed:,} sessions ({sessions_skipped:,} skipped)")

print(f"Sessions processed: {sessions_processed:,}, skipped: {sessions_skipped:,}")

# Calculate similarity scores (Jaccard-like: co-occurrence / sqrt(count1 * count2))
item_similarity = {}
MIN_COVISITS = 2  # Minimum co-visits to consider a relationship

for item1, related_items in covisit_counts.items():
    item_similarity[item1] = {}
    for item2, co_count in related_items.items():
        # Filter out weak relationships
        if co_count >= MIN_COVISITS:
            # Normalized by geometric mean of individual counts
            similarity = co_count / np.sqrt(item_counts[item1] * item_counts[item2])
            item_similarity[item1][item2] = similarity

print(f"Item similarity computed")
print(f"   Items with neighbors: {len(item_similarity):,}")
print(f"   Avg neighbors per item: {np.mean([len(v) for v in item_similarity.values()]):.1f}")

Building item-item co-visitation similarity...
  Processed 10,000 sessions (1 skipped)
  Processed 20,000 sessions (7 skipped)
  Processed 30,000 sessions (9 skipped)
  Processed 40,000 sessions (11 skipped)
  Processed 50,000 sessions (12 skipped)
  Processed 60,000 sessions (14 skipped)
  Processed 70,000 sessions (17 skipped)
  Processed 80,000 sessions (17 skipped)
  Processed 90,000 sessions (21 skipped)
  Processed 100,000 sessions (25 skipped)
  Processed 110,000 sessions (28 skipped)
  Processed 120,000 sessions (30 skipped)
  Processed 130,000 sessions (35 skipped)
  Processed 140,000 sessions (38 skipped)
  Processed 150,000 sessions (41 skipped)
  Processed 160,000 sessions (42 skipped)
  Processed 170,000 sessions (45 skipped)
  Processed 180,000 sessions (47 skipped)
  Processed 190,000 sessions (49 skipped)
  Processed 200,000 sessions (54 skipped)
  Processed 210,000 sessions (57 skipped)
  Processed 220,000 sessions (59 skipped)
  Processed 230,000 sessions (63 skipped)

In [23]:
# Save item similarity for serving
similarity_path = models_dir / 'item_similarity.pkl'
with open(similarity_path, 'wb') as f:
    pickle.dump({
        'similarity': item_similarity,
        'item_counts': dict(item_counts),
        'created_at': datetime.now().isoformat()
    }, f)
print(f"Item similarity saved: {similarity_path}")

Item similarity saved: artifacts\models\item_similarity.pkl


## Two-Stage Evaluation: Candidate Generation + Re-Ranking

In [24]:
def evaluate_two_stage(val_set, user_id_to_idx, product_id_to_idx, user_factors, item_factors,
                       popularity_scores, item_similarity, lgb_model, feature_cols, 
                       user_features, item_features, k_candidates=50, k_final=10, max_users=5000):

    idx_to_product_id = {idx: pid for pid, idx in product_id_to_idx.items()}
    
    unique_users = val_set['user_id'].unique()
    if len(unique_users) > max_users:
        print(f"Sampling {max_users} users from {len(unique_users)} for evaluation")
        sampled_users = np.random.choice(unique_users, size=max_users, replace=False)
        val_set_sample = val_set[val_set['user_id'].isin(sampled_users)]
    else:
        val_set_sample = val_set
    
    user_features_indexed = user_features.set_index('user_id')
    item_features_indexed = item_features.set_index('product_id')
    
    user_feat_cols = [col for col in user_features.columns if col != 'user_id']
    item_feat_cols = [col for col in item_features.columns if col != 'product_id']
    
    ndcg_scores = []
    recall_scores = []
    precision_scores = []
    stage1_stats = {'svd': 0, 'item_item': 0, 'popularity': 0, 'cold_start': 0}
    
    for user_id, user_data in val_set_sample.groupby('user_id'):
        if len(user_data) < 2:
            continue
        
        user_history = user_data['product_id'].values
        val_items_for_user = set(user_history)
        
        # Stage 1: Candidate generation
        candidates = set()
        
        if user_id in user_id_to_idx:
            user_idx = user_id_to_idx[user_id]
            user_vec = user_factors[user_idx]
            svd_scores = user_vec @ item_factors.T
            top_svd = np.argsort(svd_scores)[::-1][:k_candidates]
            candidates.update(
                [idx_to_product_id[idx] for idx in top_svd if idx in idx_to_product_id]
            )
            stage1_stats['svd'] += 1
        else:
            stage1_stats['cold_start'] += 1
        
        for hist_item in user_history[:5]:
            if hist_item in item_similarity:
                similar_items = sorted(
                    item_similarity[hist_item].items(),
                    key=lambda x: x[1],
                    reverse=True
                )[:20]
                candidates.update([item for item, _ in similar_items])
                stage1_stats['item_item'] += 1
        
        top_popular = popularity_scores.head(k_candidates).index.tolist()
        candidates.update(top_popular)
        stage1_stats['popularity'] += 1
        
        # Filter to only candidates that overlap with validation items for this user
        candidates = candidates & val_items_for_user
        if len(candidates) < 2:
            continue
        
        # Stage 2: Re-rank with LightGBM
        if user_id not in user_features_indexed.index:
            continue
        
        user_feat = user_features_indexed.loc[user_id]
        
        valid_candidates = [
            pid for pid in candidates if pid in item_features_indexed.index
        ]
        if len(valid_candidates) < 2:
            continue
        
        items_feat = item_features_indexed.loc[valid_candidates]
        n_candidates = len(valid_candidates)
        
        user_feat_matrix = np.tile(user_feat.values, (n_candidates, 1))
        user_feat_df = pd.DataFrame(
            user_feat_matrix,
            columns=[f'user_{col}' for col in user_feat_cols]
        )
        
        items_feat_renamed = items_feat.copy()
        items_feat_renamed.columns = [
            f'item_{col}' for col in item_feat_cols
        ]
        items_feat_renamed = items_feat_renamed.reset_index(drop=True)
        
        X_candidates = pd.concat(
            [user_feat_df, items_feat_renamed],
            axis=1
        )
        
        X_candidates['last_interaction_ts'] = 0
        X_candidates['days_since_last'] = 0
        X_candidates['interaction_count'] = 0
        X_candidates['recency_score'] = 0
        
        for col in feature_cols:
            if col not in X_candidates.columns:
                X_candidates[col] = 0
        
        X_candidates = (
            X_candidates[feature_cols]
            .apply(pd.to_numeric, errors='coerce')
            .fillna(0)
        )
        
        rerank_scores = lgb_model.predict(X_candidates)
        
        top_k_idx = np.argsort(rerank_scores)[::-1][:k_final]
        top_k_products = [valid_candidates[i] for i in top_k_idx]
        
        # Recompute relevance labels AFTER candidate filtering
        true_relevance = user_data.set_index('product_id')['relevance'].to_dict()
        y_true = [true_relevance.get(pid, 0) for pid in top_k_products]
        y_pred = [rerank_scores[i] for i in top_k_idx]
        
        if len(y_true) >= 2 and sum(y_true) > 0:
            ndcg_scores.append(
                ndcg_score([y_true], [y_pred], k=k_final)
            )
        
        relevant_items = [
            pid for pid, rel in true_relevance.items() if rel > 1
        ]
        if relevant_items:
            recall = (
                len(set(top_k_products) & set(relevant_items)) /
                len(relevant_items)
            )
            recall_scores.append(recall)
            
            precision = (
                len(set(top_k_products) & set(relevant_items)) /
                k_final
            )
            precision_scores.append(precision)
    
    print("Two-stage evaluation complete")
    print(
        f"Evaluated users: {len(ndcg_scores)}"
    )
    print(
        f"Stage 1: SVD={stage1_stats['svd']}, "
        f"Item-Item={stage1_stats['item_item']}, "
        f"Popularity={stage1_stats['popularity']}, "
        f"Cold-start={stage1_stats['cold_start']}"
    )
    
    return {
        'ndcg@10': np.mean(ndcg_scores) if ndcg_scores else 0,
        'recall@10': np.mean(recall_scores) if recall_scores else 0,
        'precision@10': np.mean(precision_scores) if precision_scores else 0,
        'stage1_stats': stage1_stats
    }

In [25]:
# Run two-stage evaluation
print("Running two-stage evaluation...")
two_stage_metrics = evaluate_two_stage(
    df_val_set, user_id_to_idx, product_id_to_idx, user_factors, item_factors,
    popularity_scores, item_similarity, lgb_model, feature_cols,
    user_features, item_features, k_candidates=50, k_final=10, max_users=5000
)

print(f"\nTwo-Stage Pipeline Metrics:")
for metric, value in two_stage_metrics.items():
    if metric != 'stage1_stats':
        print(f"  {metric}: {value:.4f}")

Running two-stage evaluation...
Sampling 5000 users from 302355 for evaluation
Two-stage evaluation complete
Evaluated users: 259
Stage 1: SVD=875, Item-Item=1989, Popularity=875, Cold-start=0

Two-Stage Pipeline Metrics:
  ndcg@10: 0.9932
  recall@10: 0.6909
  precision@10: 0.1091


In [26]:
# Save two-stage metrics
two_stage_path = models_dir / 'two_stage_metrics.json'
with open(two_stage_path, 'w') as f:
    json.dump({
        'metrics': {k: float(v) if isinstance(v, (int, float, np.number)) else v 
                   for k, v in two_stage_metrics.items()},
        'config': {
            'k_candidates': 50,
            'k_final': 10,
            'candidate_sources': ['svd', 'item_item', 'popularity']
        },
        'evaluated_at': datetime.now().isoformat()
    }, f, indent=2)
print(f"Two-stage metrics saved: {two_stage_path}")

Two-stage metrics saved: artifacts\models\two_stage_metrics.json


In [27]:
# Update comparison table with two-stage results
comparison_updated = pd.DataFrame({
    'Model': ['Popularity Baseline', 'SVD (Collaborative Filtering)', 'LightGBM Ranker', 'Two-Stage Pipeline'],
    'NDCG@10': [baseline_ndcg, svd_ndcg, lgb_metrics['ndcg@10'], two_stage_metrics['ndcg@10']],
    'Recall@10': [None, None, lgb_metrics['recall@10'], two_stage_metrics['recall@10']],
    'Precision@10': [None, None, lgb_metrics['precision@10'], two_stage_metrics['precision@10']]
})

print("\nUpdated Model Comparison:")
print(comparison_updated.to_string(index=False))

# Save comparison
comparison_path = models_dir / 'model_comparison.csv'
comparison_updated.to_csv(comparison_path, index=False)
print(f"\nComparison saved: {comparison_path}")


Updated Model Comparison:
                        Model  NDCG@10  Recall@10  Precision@10
          Popularity Baseline 0.423440        NaN           NaN
SVD (Collaborative Filtering) 0.683485        NaN           NaN
              LightGBM Ranker 0.999232   0.990792      0.472798
           Two-Stage Pipeline 0.993249   0.690909      0.109091

Comparison saved: artifacts\models\model_comparison.csv


## Execution Summary

In [31]:
from pprint import pprint
# Build comprehensive run summary from executed variables
run_summary = {
    'execution_metadata': {
        'notebook': '04_model_training.ipynb',
        'executed_at': datetime.now().isoformat(),
        'data_mode': DATA_MODE,
        'versions': {
            'pandas': pd.__version__,
            'numpy': np.__version__,
            'lightgbm': lgb.__version__
        }
    },
    
    'dataset_statistics': {
        'events_loaded': len(df_events),
        'unique_users': int(df_events['user_id'].nunique()),
        'unique_products': int(df_events['product_id'].nunique()),
        'event_type_counts': {str(k): int(v) for k, v in df_events['event_type'].value_counts().to_dict().items()},
        'training_samples': len(df_train_set),
        'validation_samples': len(df_val_set),
        'train_split_pct': round(len(df_train_set)/(len(df_train_set)+len(df_val_set))*100, 1),
        'val_split_pct': round(len(df_val_set)/(len(df_train_set)+len(df_val_set))*100, 1),
        'user_features_count': len(user_features),
        'item_features_count': len(item_features),
        'interaction_features_count': len(interaction_features)
    },
    
    'models_evaluated': {
        'popularity_baseline': {
            'ndcg@10': round(float(baseline_ndcg), 4)
        },
        'svd_collaborative_filtering': {
            'n_components': n_components,
            'explained_variance': round(float(svd_model.explained_variance_ratio_.sum()), 4),
            'matrix_shape': list(interaction_matrix_sparse.shape),
            'matrix_nnz': int(interaction_matrix_sparse.nnz),
            'sparsity_pct': round(100 * (1 - interaction_matrix_sparse.nnz / (interaction_matrix_sparse.shape[0] * interaction_matrix_sparse.shape[1])), 4),
            'memory_mb': round(interaction_matrix_sparse.data.nbytes / 1024 / 1024, 2),
            'ndcg@10': round(float(svd_ndcg), 4)
        },
        'lightgbm_ranker': {
            'n_features': len(feature_cols),
            'num_boost_round': 100,
            'ndcg@10': round(float(lgb_metrics['ndcg@10']), 4),
            'recall@10': round(float(lgb_metrics['recall@10']), 4),
            'precision@10': round(float(lgb_metrics['precision@10']), 4),
            'offline_metric_caveat': True,
            'caveat_reason': 'Strong purchase signals in training data + offline evaluation may overestimate real-world performance',
            'top_5_features': [
                {'feature': row['feature'], 'importance': round(row['importance'], 2)} 
                for _, row in feature_importance.head(5).iterrows()
            ]
        },
        'item_item_similarity': {
            'items_with_neighbors': len(item_similarity),
            'avg_neighbors_per_item': round(float(np.mean([len(v) for v in item_similarity.values()])), 1)
        },
        'two_stage_pipeline': {
            'k_candidates': 50,
            'k_final': 10,
            'ndcg@10': round(float(two_stage_metrics['ndcg@10']), 4),
            'offline_metric_caveat': True,
            'caveat_reason': 'Strong purchase signals in training data + offline evaluation may overestimate real-world performance',
            'stage1_usage': two_stage_metrics['stage1_stats']
        }
    },
    
    'metric_comparison': {
        'best_ndcg@10': round(max(baseline_ndcg, svd_ndcg, lgb_metrics['ndcg@10'], two_stage_metrics['ndcg@10']), 4),
        'lightgbm_improvement_over_baseline_pct': round((lgb_metrics['ndcg@10'] - baseline_ndcg) / baseline_ndcg * 100, 2),
        'two_stage_improvement_over_baseline_pct': round((two_stage_metrics['ndcg@10'] - baseline_ndcg) / baseline_ndcg * 100, 2),
        'two_stage_vs_lightgbm_delta': round(two_stage_metrics['ndcg@10'] - lgb_metrics['ndcg@10'], 4),
        'memory_saved_gb': round(interaction_matrix_sparse.shape[0] * interaction_matrix_sparse.shape[1] * 8 / 1024 / 1024 / 1024 - interaction_matrix_sparse.data.nbytes / 1024 / 1024 / 1024, 2)
    },
    
    'artifacts_saved': {
        'models_directory': str(models_dir),
        'files_created': [
            {'name': 'lightgbm_ranker.txt', 'exists': os.path.exists(models_dir / 'lightgbm_ranker.txt')},
            {'name': 'model_metadata.json', 'exists': os.path.exists(models_dir / 'model_metadata.json')},
            {'name': 'svd_model.pkl', 'exists': os.path.exists(models_dir / 'svd_model.pkl')},
            {'name': 'item_similarity.pkl', 'exists': os.path.exists(models_dir / 'item_similarity.pkl')},
            {'name': 'feature_importance.csv', 'exists': os.path.exists(models_dir / 'feature_importance.csv')},
            {'name': 'model_comparison.csv', 'exists': os.path.exists(models_dir / 'model_comparison.csv')},
            {'name': 'two_stage_metrics.json', 'exists': os.path.exists(models_dir / 'two_stage_metrics.json')}
        ]
    }
}

print("EXECUTION SUMMARY")
pprint(run_summary, width=120, compact=False)

EXECUTION SUMMARY
{'artifacts_saved': {'files_created': [{'exists': True, 'name': 'lightgbm_ranker.txt'},
                                       {'exists': True, 'name': 'model_metadata.json'},
                                       {'exists': True, 'name': 'svd_model.pkl'},
                                       {'exists': True, 'name': 'item_similarity.pkl'},
                                       {'exists': True, 'name': 'feature_importance.csv'},
                                       {'exists': True, 'name': 'model_comparison.csv'},
                                       {'exists': True, 'name': 'two_stage_metrics.json'}],
                     'models_directory': 'artifacts\\models'},
 'dataset_statistics': {'event_type_counts': {'add_to_cart': 69332, 'purchase': 22457, 'view': 2664312},
                        'events_loaded': 2756101,
                        'interaction_features_count': 2145179,
                        'item_features_count': 235061,
                        'tra

In [32]:
# Save run summary
summary_path = models_dir / 'run_summary.json'
with open(summary_path, 'w') as f:
    json.dump(run_summary, f, indent=2)

print(f"\nRun summary saved: {summary_path}")


Run summary saved: artifacts\models\run_summary.json
